In [0]:
spark.sql("USE CATALOG workspace")

from pyspark.sql.functions import (
    col, trim, lit, when, current_timestamp,
    datediff, round
)


bronze_df     = spark.table("bronze.bench_periods")
silver_emp_df = spark.table("silver.employees").select(
    "employee_id", "seniority"
)

print(f"Bronze bench periods: {bronze_df.count()}")
print(f"Silver employees:     {silver_emp_df.count()}")
bronze_df.printSchema()

In [0]:
# ── Apply cleaning transformations ────────────────────────────────────────────
valid_emp_ids = [row.employee_id for row in silver_emp_df.collect()]

# Estimated daily cost by seniority (£)
DAILY_COST_MAP = {
    "Analyst":           250,
    "Consultant":        350,
    "Senior Consultant": 450,
    "Manager":           600,
    "Senior Manager":    800
}

# Join seniority from silver employees
cleaned_df = bronze_df.join(silver_emp_df, on="employee_id", how="left")

cleaned_df = (
    cleaned_df
    .withColumn("reason", trim(col("reason")))
    .withColumn("employee_valid", when(
        col("employee_id").isin(valid_emp_ids),
        lit(True)).otherwise(lit(False)))
    .withColumn("date_valid", when(
        col("end_date") > col("start_date"),
        lit(True)).otherwise(lit(False)))
    .withColumn("days_on_bench_clean", datediff(
        col("end_date"), col("start_date")))
    .withColumn("excessive_bench", when(
        col("days_on_bench_clean") > 90,
        lit(True)).otherwise(lit(False)))
    .withColumn("unexpected_bench", when(
        col("seniority").isin(["Manager", "Senior Manager"]),
        lit(True)).otherwise(lit(False)))
    .withColumn("estimated_daily_cost", when(
        col("seniority") == "Analyst",           lit(DAILY_COST_MAP["Analyst"]))
        .when(col("seniority") == "Consultant",        lit(DAILY_COST_MAP["Consultant"]))
        .when(col("seniority") == "Senior Consultant", lit(DAILY_COST_MAP["Senior Consultant"]))
        .when(col("seniority") == "Manager",           lit(DAILY_COST_MAP["Manager"]))
        .when(col("seniority") == "Senior Manager",    lit(DAILY_COST_MAP["Senior Manager"]))
        .otherwise(lit(0)))
    .withColumn("estimated_bench_cost", round(
        col("days_on_bench_clean") * col("estimated_daily_cost"), 2))
    .withColumn("has_nulls", when(
        col("bench_id").isNull() |
        col("employee_id").isNull() |
        col("start_date").isNull() |
        col("end_date").isNull(),
        lit(True)).otherwise(lit(False)))
    .withColumn("silver_loaded_at", current_timestamp())
    .withColumn("source_table",     lit("bronze.bench_periods"))
)

# Drop duplicates on bench_id
cleaned_df = cleaned_df.dropDuplicates(["bench_id"])

print(f"Silver row count:          {cleaned_df.count()}")
print(f"Invalid employee IDs:      {cleaned_df.filter(col('employee_valid') == False).count()}")
print(f"Invalid dates:             {cleaned_df.filter(col('date_valid') == False).count()}")
print(f"Excessive bench periods:   {cleaned_df.filter(col('excessive_bench') == True).count()}")
print(f"Unexpected bench:          {cleaned_df.filter(col('unexpected_bench') == True).count()}")
print(f"Rows with nulls:           {cleaned_df.filter(col('has_nulls') == True).count()}")
cleaned_df.show(5)

In [0]:
(
    cleaned_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.bench_periods")
)

print(f"Written to silver.bench_periods: {spark.table('silver.bench_periods').count()} rows")

In [0]:
spark.sql("USE CATALOG workspace")

tables = [
    "employees",
    "projects",
    "assignments",
    "timesheets",
    "skills",
    "bench_periods"
]

print("=== Silver Layer Summary ===\n")
for table in tables:
    count = spark.sql(f"SELECT COUNT(*) as cnt FROM silver.{table}").collect()[0][0]
    print(f"silver.{table}: {count} rows")